In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd

2026-02-28 15:22:21.372159: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-28 15:22:21.428747: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-28 15:22:21.429625: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-28 15:22:22.128453: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### **Carga y limpieza de datos:**

Dentro del conjunto de datos seleccionado, las variables *transaction_id* y *user_id* se consideran irrelevantes para el entrenamiento del modelo. En el caso de transaction_id, su inclusión podría inducir overfitting, ya que se trata de un identificador único sin valor predictivo real. Por su parte, user_id requeriría un volumen significativamente mayor de información histórica para modelar patrones de reincidencia de manera confiable, por lo que también se excluye del análisis.

Las variables *(transaction_amount, account_age_days, transaction_hour, previous_failed_attempts, avg_transaction_amount, ip_risk_score, login_attempts_last_24h)* corresponden a datos numéricos continuos o discretos, los cuales son directamente utilizables y evaluables por la red neuronal tras un proceso adecuado de normalización o estandarización.

En contraste, las variables *(transaction_type, payment_mode, device_type, device_location)* son de tipo categórico (string). Para poder incorporarlas al modelo, se transforman mediante un mapeo numérico apropiado, permitiendo su procesamiento dentro de la arquitectura neuronal.

La variable *is_international* es de naturaleza binaria y puede ser utilizada directamente como característica de entrada, ya que representa información relevante en formato compatible con el modelo.

En consecuencia, todas las variables mencionadas constituyen los features del modelo. La variable objetivo se encuentra en la última columna bajo el nombre *fraud_label*, la cual representa una salida binaria que indica si la transacción corresponde o no a un caso de fraude.

In [2]:
df=pd.read_csv('Datasets/Digital_Payment_Fraud_Detection_Dataset.csv')
#cols_str= [transaction_type, payment_mode, device_type, device_location]
#print(df.dtypes)
df=df.drop(columns=['transaction_id', 'user_id'])

for i in df.columns:
    if df[i].dtype== object:
        unicos=df[i].unique()
        mapeo={}
        c=0
        for j in unicos:
            mapeo[j]=c
            c+=1
        #print(i, df[i].dtype, mapeo)
        df[f'{i}_cat']=df[i].replace(mapeo)
        df=df.drop(columns=i)

df.head(2)

,transaction_amount,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label,transaction_type_cat,payment_mode_cat,device_type_cat,device_location_cat
0,18758.28,895,14,1,25535.84,0,0.718,4,0,0,0,0,0
1,47538.18,918,21,0,3955.85,0,0.525,9,0,1,1,1,0


### **Datos de ensayo y verificación**

In [3]:
rand_sel=np.random.rand(len(df))*7
rand_sel=rand_sel.astype(int)
df['random']=rand_sel
df_princ=df[df['random']<4].copy().drop(columns='random')
df_1=df[df['random']==4].copy().drop(columns='random')
df_2=df[df['random']==5].copy().drop(columns='random')
df_3=df[df['random']==6].copy().drop(columns='random')

In [4]:
print(  len(df_princ[df_princ['fraud_label']==1]),
        len(df_princ[df_princ['fraud_label']==0]), '\n',
        len(df_1[df_1['fraud_label']==1]), 
        len(df_1[df_1['fraud_label']==0]), '\n',
        len(df_2[df_2['fraud_label']==1]), 
        len(df_2[df_2['fraud_label']==0]), '\n',
        len(df_3[df_3['fraud_label']==1]),
        len(df_3[df_3['fraud_label']==0]))

290 3972 
 59 1039 
 74 1004 
 66 996


### **Definición del modelo:**

Se procede a definir el modelo, con los datos categorizados y la unica salida binaria

In [5]:
x=[]
for i in df_princ.columns:
    if i!= 'fraud_label':
        x.append(df_princ[i])

X=np.column_stack(x)
tamaño= len(x)

In [44]:
entrada = tf.keras.layers.Dense(units=tamaño, input_shape=[tamaño])
c1 = tf.keras.layers.Dense(units=tamaño)
c2 = tf.keras.layers.Dense(units=tamaño)
c3 = tf.keras.layers.Dense(units=tamaño)
salida = tf.keras.layers.Dense(units=1)
red = tf.keras.Sequential([entrada, c1, c2, c3, salida])
red.compile(
    optimizer=tf.keras.optimizers.Adam(0.1),
    loss='mean_squared_error'
)

In [ ]:
historial = red.fit(X, df_princ['fraud_label'], epochs=10000, verbose=False)

In [37]:
fila=25
cols_ver=['transaction_amount', 'account_age_days', 'transaction_hour', 'previous_failed_attempts', 'avg_transaction_amount', 'is_international', 'ip_risk_score', 'login_attempts_last_24h', 'transaction_type_cat', 'payment_mode_cat','device_type_cat','device_location_cat']
k=df_1[cols_ver].iloc[[fila]].values

k2=df_1[['fraud_label']].iloc[fila].values
print(k,'\n'*2, k2)
red.predict(k)

[[3.049774e+04 1.811000e+03 1.100000e+01 3.000000e+00 4.199500e+02
  0.000000e+00 3.330000e-01 9.000000e+00 0.000000e+00 1.000000e+00
  0.000000e+00 0.000000e+00]] 

 [1]
1/1 [==============================] - 0s 14ms/step


array([[0.17880355]], dtype=float32)

In [43]:

def clasificar_dataframe(df, modelo, columnas):
    X = df[columnas].values
    preds = modelo.predict(X)
    df['clasificacion'] = preds#(preds > 0.5).astype(int)
    return df

df_1= clasificar_dataframe(df_1, red, cols_ver)

df_1[df_1['clasificacion']<-0.5].reset_index().head(50)

35/35 [==============================] - 0s 706us/step


,index,transaction_amount,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label,transaction_type_cat,payment_mode_cat,device_type_cat,device_location_cat,clasificacion
0,56,4470.20,313,5,3,29963.03,0,0.937,9,0,1,2,1,2,-0.643615
1,119,26990.14,1992,15,4,28030.84,0,0.300,9,0,1,3,1,0,-0.505986
2,185,33016.20,1044,5,3,29438.33,0,0.984,1,0,0,3,1,2,-0.541695
3,188,26506.05,1989,20,3,29320.95,0,0.826,7,0,2,0,0,4,-0.539786
4,255,27862.22,922,0,2,29744.29,0,0.854,3,1,1,1,1,2,-0.565128
5,291,1915.54,1489,6,2,26902.69,0,0.211,6,0,2,0,0,2,-0.553849
6,332,768.95,1648,23,1,25641.68,0,0.295,6,0,1,0,1,1,-0.523427
7,339,4936.82,467,3,2,27168.06,0,0.157,3,0,0,3,1,2,-0.570630
8,345,30811.71,241,14,3,29279.81,0,0.980,9,0,2,0,1,2,-0.559455
9,434,4251.13,151,13,3,24239.34,0,0.400,8,0,1,2,2,3,-0.507241
